# Entrenamiento del encoder Deep Sets (N-invariante)

Flujo **datos → transformación → entrenamiento**, paso a paso.

- **Una arquitectura por orden** $n_x$: como el orden cambia la dimensión por vértice
  $d_v=n_x^2+n_x n_u$ y la salida $\dim y$, se construye un modelo distinto por cada $n_x$.
- **Cada modelo es N-invariante**: el mismo modelo cubre cualquier número de vértices $N$.
- Las **pruebas se reportan separadas por cantidad de vértices** $N$.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch
from entrenamiento import training as T
torch.set_num_threads(6)

# --- Configuración ---
ORDERS      = [2, 3, 4, 5]   # n_x : una arquitectura por cada uno
VERTICES    = [2, 3, 4, 5]   # N   : el mismo modelo cubre todos
INPUTS      = 1              # n_u
ALPHA       = 0.01
EPOCHS      = 15             # súbelo (p. ej. 50) para resultados finales
DR_TRAIN    = 30
DR_EVAL     = 1500
BATCH       = 16
SEED        = 42
LIMIT_PER_N = 150           # usa None para los 500 sistemas por (orden, N)
DEVICE      = "cpu"
print("Configuración lista.")

Configuración lista.


## Paso 1 — Carga y transformación de datos

Cada vértice $(A_i,B_i)$ se aplana a un vector $x_i=[\operatorname{vec}(A_i),\operatorname{vec}(B_i)]\in\mathbb{R}^{d_v}$,
y el sistema completo (politopo) es el **conjunto** de esos $N$ vectores: un tensor $(N, d_v)$.
El encoder Deep Sets arma esa vista internamente desde `A_poly`, `B_poly`.

In [ ]:
# Cargamos un caso de ejemplo (orden 3, N=4) y mostramos la transformación
items = T.load_vertices(order=3, inputs=INPUTS, vertices=4, limit=3)
A0, B0 = items[0]
N = A0.shape[0]
print("Un sistema (orden 3, N=4):  A", tuple(A0.shape), " B", tuple(B0.shape))

# Vista de conjunto que recibe el encoder: (N, d_v)
x = torch.cat([A0.reshape(N, -1), B0.reshape(N, -1)], dim=-1)
dv = x.shape[1]
print(f"Conjunto por-vértice (N, d_v) = {tuple(x.shape)}   d_v = n_x^2 + n_x*n_u = {dv}")
torch.set_printoptions(precision=4, sci_mode=False)
print(x)

## Paso 2 — Una arquitectura por orden

Construimos un modelo Deep Sets por cada orden. Nota cómo $d_v$, $\dim y$ y el número de
parámetros cambian con el orden (por eso es una arquitectura por orden), mientras que dentro
de un orden el modelo sirve para cualquier $N$.

In [ ]:
models = {o: T.build_model(order=o, inputs=INPUTS, alpha=ALPHA, dr_iters=DR_TRAIN)
          for o in ORDERS}

print(f"{'orden n_x':>9} | {'d_v':>4} | {'dim_y':>5} | {'parámetros':>11}")
print("-" * 40)
for o in ORDERS:
    m = models[o]
    npar = sum(p.numel() for p in m.parameters())
    print(f"{o:>9} | {m.vertex_dim:>4} | {m.dim_y:>5} | {npar:>11,}")

## Paso 3 — Preparar particiones y rutina por orden

`preparar(order)` carga y parte (80/20) los datos de **todos los $N$** de ese orden.
`correr_orden(order)` entrena la arquitectura de ese orden sobre todos los $N$ y luego
**evalúa por separado para cada cantidad de vértices**.

In [ ]:
def preparar(order):
    by_N = {}
    for N in VERTICES:
        data = T.load_vertices(order, INPUTS, N, limit=LIMIT_PER_N)
        by_N[N] = T.split_items(data, seed=SEED)   # -> (train, test)
    return by_N

RESULTS = {}

def correr_orden(order):
    by_N = preparar(order)
    train_by_N = {N: tr for N, (tr, te) in by_N.items()}
    print(f"=== Orden n_x={order}: entrenando UNA arquitectura sobre N={VERTICES} ===")
    T.train(models[order], train_by_N, epochs=EPOCHS, batch=BATCH,
            seed=SEED, device=DEVICE)

    print(f"\nEvaluación separada por número de vértices (orden {order}):")
    print(f"{'N':>3} | {'estable%':>8} | {'decay%':>7} | {'peor-eig':>9} | {'n':>4}")
    print("-" * 44)
    RESULTS[order] = {}
    for N, (tr, te) in by_N.items():
        r = T.evaluate(models[order], te, order=order, alpha=ALPHA,
                       dr_eval=DR_EVAL, device=DEVICE)
        RESULTS[order][N] = r
        print(f"{N:>3} | {r['stable_pct']:>7.0f} | {r['decay_pct']:>6.0f} | "
              f"{r['worst_mean']:>+9.3f} | {r['n']:>4}")

print("Rutinas listas.")

### Orden $n_x=2$  —  entrenamiento + pruebas por nº de vértices

In [ ]:
correr_orden(2)

### Orden $n_x=3$  —  entrenamiento + pruebas por nº de vértices

In [ ]:
correr_orden(3)

### Orden $n_x=4$  —  entrenamiento + pruebas por nº de vértices

In [ ]:
correr_orden(4)

### Orden $n_x=5$  —  entrenamiento + pruebas por nº de vértices

In [ ]:
correr_orden(5)

## Resumen final — estabilización (%) por orden y número de vértices

Filas = orden $n_x$ (una arquitectura por fila); columnas = nº de vértices $N$
(un mismo modelo cubre toda la fila).

In [ ]:
print(f"{'n_x \\ N':>8} | " + " | ".join(f"N={N:>1}" for N in VERTICES))
print("-" * (10 + 7 * len(VERTICES)))
for o in ORDERS:
    if o not in RESULTS:
        continue
    cells_ = " | ".join(f"{RESULTS[o][N]['stable_pct']:>3.0f}%" for N in VERTICES)
    print(f"{o:>8} | {cells_}")